# Random Forest Code Companion

This notebook shows the code side of Random Forest. The goal is to connect the theory to practice: many trees, bootstrap sampling, random feature selection, voting, OOB score, feature importance, and evaluation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score


## Load a Classification Dataset

Random Forest combines many Decision Trees. Here the task is binary classification.


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print("Classes:", dict(enumerate(cancer.target_names)))
X.head()


## Split the Data

The forest learns from training data and is evaluated on test data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## Train the Forest

`n_estimators` is the number of trees. `bootstrap=True` means each tree trains on a bootstrap sample. `max_features="sqrt"` means each split sees only a random subset of features. `oob_score=True` allows out-of-bag evaluation.


In [ ]:
forest = RandomForestClassifier(
    n_estimators=200,
    criterion="gini",
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

forest.fit(X_train, y_train)


## Out-of-Bag Score

OOB score estimates performance using rows that were not included in each tree's bootstrap sample.


In [ ]:
print(f"OOB score: {forest.oob_score_:.3f}")


## Predict with Voting

For classification, every tree votes for a class. The class with the most votes becomes the Random Forest prediction.


In [ ]:
y_pred = forest.predict(X_test)
y_proba = forest.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "actual": y_test.map(dict(enumerate(cancer.target_names))),
    "predicted": pd.Series(y_pred, index=y_test.index).map(dict(enumerate(cancer.target_names))),
    "probability_class_1": y_proba
})

results.head(10)


## Evaluate the Forest

The same classification metrics used for Logistic Regression and Decision Trees also apply to Random Forest.


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=cancer.target_names))


## Feature Importance

Random Forest averages feature importance across many trees, so it is usually more stable than feature importance from one Decision Tree.


In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": forest.feature_importances_
}).sort_values("importance", ascending=False)

importance.head(10)


In [ ]:
top_features = importance.head(10).sort_values("importance")

plt.figure(figsize=(8, 5))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()
